# DermaVision AI — Module 5: Variational Autoencoder (VAE)
### Compact Latent Representations, Clinical Similarity Search & Latent Space Visualization

**Authors:** DermaVision AI Team (Module 5)  
**Curriculum Mapping:** *"VAE learns compact representations of skin images for similarity search and visualization."*

---

### 1. Overview & Clinical Context
Skin lesions exhibit complex morphological patterns (pigment asymmetry, border irregularity, atypical network reticulation). In this module, we evaluate a **Convolutional Variational Autoencoder (ConvVAE)** that compresses $64 \times 64 \times 3$ RGB skin lesion images into a continuous 32-dimensional latent space ($z \in \mathbb{R}^{32}$).

Using this latent space, we build a **Case-Based Reasoning (CBR) Similarity Engine** to retrieve matching historical clinical cases for decision support.

In [ ]:
import os
import sys
import torch
import matplotlib.pyplot as plt
from PIL import Image

# Add project root
sys.path.insert(0, os.path.abspath('..'))
if not os.path.exists('src'):
    sys.path.insert(0, os.path.abspath('.'))

from src.module5_vae import (
    SkinLesionVAE,
    SkinCaseSimilarityEngine,
    plot_reconstruction_grid,
    plot_latent_interpolation
)

device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f"DermaVision VAE Engine Initialized on: {device.upper()}")

### 2. Loading Trained ConvVAE Model
We instantiate the `SkinLesionVAE` with latent dimension $z = 32$ and load trained checkpoint weights.

In [ ]:
latent_dim = 32
model = SkinLesionVAE(latent_dim=latent_dim).to(device)
vae_weights = os.path.join('models', 'module5_vae_best.pth')

if os.path.exists(vae_weights):
    model.load_state_dict(torch.load(vae_weights, map_location=device))
    print(f"Successfully loaded trained VAE checkpoint from: {vae_weights}")
else:
    print(f"Warning: {vae_weights} not found. Running with random initialization.")

model.eval()

### 3. Morphological Lesion Reconstruction
We evaluate how accurately the VAE reconstructs skin lesions. The top row displays original clinical lesions, and the bottom row displays VAE decodings from the latent mean $\mu$.

In [ ]:
recon_plot = "Module5_VAE_Reconstruction_Grid.png"
if os.path.exists(recon_plot):
    plt.figure(figsize=(14, 5))
    im = Image.open(recon_plot)
    plt.imshow(im)
    plt.axis('off')
    plt.title("DermaVision AI — VAE Lesion Reconstructions (Top: Original | Bottom: Reconstructed)", fontsize=14, fontweight='bold')
    plt.show()
else:
    print(f"{recon_plot} will be generated after running train_module5_vae.py")

### 4. Latent Space 2D Manifold Visualization
We project the 32-dimensional latent vectors into 2D using **t-SNE**, showing clustering of distinct skin conditions (`ACK`, `BCC`, `MEL`, `NEV`, `SCC`, `SEK`).

In [ ]:
tsne_plot = "Module5_VAE_Latent_Space_tSNE.png"
if os.path.exists(tsne_plot):
    plt.figure(figsize=(10, 8))
    im = Image.open(tsne_plot)
    plt.imshow(im)
    plt.axis('off')
    plt.title("DermaVision AI — 2D Latent Space Projection (t-SNE)", fontsize=14, fontweight='bold')
    plt.show()
else:
    print(f"{tsne_plot} will be generated after training")

### 5. Clinical Case Similarity Search (Case-Based Reasoning)
We load the reference gallery (`models/module5_vae_case_gallery.pt`) containing 150 diagnosed clinical cases with verified histological outcomes.

When a query skin image is provided, the engine computes **Cosine Similarity** and **Euclidean Distance** in the VAE latent space to retrieve the top-3 closest matching cases.

In [ ]:
engine = SkinCaseSimilarityEngine(model, device=device)
gallery_path = os.path.join('models', 'module5_vae_case_gallery.pt')

if os.path.exists(gallery_path):
    engine.load_gallery(gallery_path)
    
    # Query with a sample lesion from the indexed gallery
    sample_tensor = torch.rand(3, 64, 64) # Simulated query patient image
    query_results = engine.find_similar_cases(sample_tensor, top_k=3)
    
    print(f"=== CLINICAL SIMILARITY SEARCH RESULTS ===")
    print(f"Query Anomaly Score: {query_results['query_anomaly_score']} ({query_results['anomaly_interpretation']})\n")
    
    for rank, case in enumerate(query_results['top_similar_cases'], 1):
        print(f"Match #{rank}:")
        print(f" - Case ID: {case['case_id']}")
        print(f" - Diagnosis: {case['diagnosis_code']} ({case['diagnosis_name']})")
        print(f" - Similarity Score: {case['similarity_score']:.2f}%")
        print(f" - Latent Distance (L2): {case['latent_distance']}")
        print(f" - Risk Level: {case['risk_level']} | Biopsied: {case['biopsied']}")
        print(f" - Historical Outcome: {case['clinical_outcome']}\n")
else:
    print(f"Gallery {gallery_path} not found. Run train_module5_vae.py to generate gallery.")

### 6. Continuous Latent Manifold Traversal (Lesion Interpolation)
Linear interpolation between two points in latent space: $z(\alpha) = (1 - \alpha) z_1 + \alpha z_2$.
Demonstrates smooth morphological transition from benign to malignant melanocytic structures.

In [ ]:
interp_plot = "Module5_VAE_Latent_Interpolation.png"
if os.path.exists(interp_plot):
    plt.figure(figsize=(14, 3.5))
    im = Image.open(interp_plot)
    plt.imshow(im)
    plt.axis('off')
    plt.title("DermaVision AI — Continuous Latent Manifold Traversal", fontsize=14, fontweight='bold')
    plt.show()
else:
    print(f"{interp_plot} will be generated after training")

### 7. Conclusion & Integration with Module 5
- Successfully demonstrated **Convolutional Variational Autoencoder (ConvVAE)** representation learning.
- Enabled **Case-Based Reasoning (CBR) Similarity Search** in clinical decision-support.
- Directly fulfilled the curriculum requirement for compact representations and visualization.